# Repository integration
Run as a separate Databricks ingestion Job, never from the chat request.
Upload the repository including `contract_agent_config.py`. Set `config_dir` to its workspace root and `config_path` to the reviewed dedup JSON.
Existing pipeline table schemas and the Vector Search endpoint are prerequisites. Use one concurrent Job run: staging tables are shared.
Removed/superseded documents are queued in `files_pending_removal`, not deleted from the index.


# Parsing and Indexing Pipeline



| Step | Output |
| --- | --- |
| 1. Parse (ai_parse_document) | `dev_diamonddaas.aia_contract_chatbot.brz_parsed_document` |
| 2. Metadata (ai_extract) | `dev_diamonddaas.aia_contract_chatbot.document_ai_metadata` |
| 3. Chunks | `dev_diamonddaas.aia_contract_chatbot.document_chunk` |
| 3b/4/4b. Contextualized chunks + validated PDF links | `dev_diamonddaas.aia_contract_chatbot.ai_contextualized_chunk` |
| 5. Embeddings(databricks-gte-large-en) | `dev_diamonddaas.aia_contract_chatbot.ai_contextualized_chunk_embedding` |
| 6. Vector Search index | `dev_diamonddaas.aia_contract_chatbot.ai_contextualized_chunk_index` (endpoint `aia_contract_search`) |


### 1. What the pipeline does

| Step | Function | What it does | Runs only when | Output table |
| --- | --- | --- | --- | --- |
| 0. File check | `check_document_type`, `set_files` | Name must end in `.pdf` and file content must start with `%PDF`; file must exist in the source folder. Non-PDFs return "Unsupported file format" | Always | Temp view `_files` |
| 1. Parse | `parse_documents` | `ai_parse_document` (v2.0) turns the PDF into pages + elements (text, titles, tables, headers...) and records page/element counts and errors | File is new or its modification time changed (or `force=True`) | `contract_management_bronze.brz_parsed_document` |
| 2. Metadata | `extract_metadata` | `ai_extract` (v2.1, with confidence scores) pulls 20 contract fields from the parsed document | File is new or was re-parsed | `contract_management_silver.document_ai_metadata` |
| 3. Chunk | `chunk_documents` | Section-aware chunking of parsed elements (rules in section 3) | File is new, re-parsed, or `REBUILD_CHUNKS = True` | `contract_management_silver.document_chunk` |
| 3b. Validate links | `validate_doc_links` | Builds one link per PDF, opens the link and the file behind it, checks it is a real PDF | Always | Temp view `_link_check` |
| 4. Contextualize | `contextualize_chunks` | Prepends a metadata header to each chunk and adds validated links | Chunks or metadata changed | `contract_management_ai.ai_contextualized_chunk` |
| 4b. Refresh links | `refresh_links` | Re-applies validated links to existing rows (only changed rows are updated) | Always | `contract_management_ai.ai_contextualized_chunk` |
| 5. Embed | `embed_chunks` | Syncs rows to the embedding table and embeds only new or changed `contextualized_text` with `databricks-gte-large-en` (1024-dim); removes chunks that no longer exist | Row hash changed | `contract_management_ai.ai_contextualized_chunk_embedding` |
| 6. Index | `sync_vector_index` | Creates (if missing) and triggers a sync of the Delta Sync Vector Search index on endpoint `aia_contract_search` | Index is behind the table | `contract_management_ai.ai_contextualized_chunk_index` |


### 2. Metadata extracted (Step 2, `ai_extract`)

| Field | Type | Description |
| --- | --- | --- |
| `document_title` | string | Title on the first page |
| `document_type` | enum | Master Agreement, License Agreement, Subscription Agreement, Order Form, Pricing Schedule, Service Schedule, Amendment, Addendum, Purchase Agreement, Other |
| `contract_reference` | string | Contract / order form / schedule reference number |
| `parent_agreement_reference` | string | Master or parent agreement this document falls under |
| `vendor_name` | string | Vendor, data provider or licensor |
| `aia_entity` | string | AIA contracting entity (client / licensee) |
| `effective_date` | date | Start date |
| `expiry_date` | date | End / expiry date |
| `signed_date` | date | Latest signature date |
| `contract_term` | string | Duration, e.g. 36 months |
| `auto_renewal` | boolean | Renews automatically |
| `renewal_terms` | string | Renewal terms and price uplift |
| `termination_notice_period` | string | Notice needed to terminate / not renew |
| `governing_law` | string | Governing law jurisdiction |
| `currency` | string | ISO 4217 currency code |
| `total_contract_value` | double | Total (or annual) fees |
| `payment_terms` | string | Invoicing frequency and payment terms |
| `products_services` | array<string> | Products, datasets, indices or services licensed |
| `permitted_users_or_sites` | string | Permitted users, locations, usage restrictions |
| `signatories` | array<struct<name, title, party>> | People who signed |
| `extraction`, `extraction_error`, `extracted_at` | variant, string, timestamp | Raw `ai_extract` output, error (if any), run time |


### 3. Chunking rules (Step 3)

| Rule | Detail |
| --- | --- |
| Sections | Each `title` / `section_header` element starts a new section; its text becomes `section_title` ("Document start" before the first heading) |
| Tables | Every table is its own chunk and is never split |
| Text packing | Text elements in a section are packed up to `MAX_CHUNK_CHARS = 2000` (~500 tokens); a single long element is never split |
| Dropped | `page_header`, `page_footer`, `page_number`, empty elements, and heading-only chunks |
| `chunk_id` | `md5(file_path)` + `_` + 4-digit `chunk_index` (stable per file) |
| Page range | `page_start` / `page_end` (1-based) from element bounding boxes |
| Other columns | `section_id`, `element_types`, `chunk_text`, `chunk_chars`, `chunked_at`, `doc_uri` |

### 4. Hyperlinks and citations (Steps 3b, 4, 4b)

| Column | Example / format | Meaning |
| --- | --- | --- |
| `doc_uri` | `/Volumes/dev_diamonddaas/raw/aia/.../file.pdf` | Volume path of the PDF (`dbfs:` prefix removed) |
| `source_uri` | `/Volumes/.../My%20File.pdf` | URL-encoded volume path |
| `source_url` | `https://adb-3456006072984226.6.azuredatabricks.net/explore/data/volumes/<catalog>/<schema>/<volume>?o=3456006072984226&volumePath=<encoded path>` | Clickable Catalog Explorer link to the PDF (one per file, no page number) |
| `citation` | `[file name \| section title](source_url)` | Markdown citation for agent answers |

### 5. Contextualized chunk (Step 4)

`contextualized_text` = `context_header` + blank line + `chunk_text`. The header contains, one per line:

| Header line | Source |
| --- | --- |
| Document | `document_title` (or file name) + file name |
| Document type | `document_type` |
| Vendor | `vendor_name` |
| AIA entity | `aia_entity` |
| Contract reference | `contract_reference` |
| Parent agreement | `parent_agreement_reference` |
| Term | `effective_date` to `expiry_date` |
| Products/services | `products_services` |
| Section | `section_title` |
| Pages | `page_start`-`page_end` |
| Source URI | `doc_uri` |

`contextualized_text` is what gets embedded, so searches match on both the chunk content and its contract metadata.

### 6. Embedding and Vector Search index (Steps 5, 6)

| Item | Value |
| --- | --- |
| Embedding model | `databricks-gte-large-en` (1024 dimensions, 8192-token context) via `ai_query` |
| Index | `dev_diamonddaas.contract_management_ai.ai_contextualized_chunk_index`, Delta Sync, triggered, primary key `chunk_id` |
| Endpoint | `aia_contract_search` |
| Synced columns | `chunk_id`, `file_path`, `file_name`, `chunk_index`, `section_title`, `page_start`, `page_end`, `document_title`, `document_type`, `contract_reference`, `vendor_name`, `aia_entity`, `effective_date`, `expiry_date`, `chunk_text`, `contextualized_text`, `doc_uri`, `source_uri`, `source_url`, `citation` |


In [ ]:

# Parsing and Indexing Pipeline
#===================================================================================
import re
import time
import urllib.parse
import requests
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound, ResourceDoesNotExist, BadRequest
from databricks.sdk.service import vectorsearch as vs
import sys
import importlib

# Load the repository's non-secret config; authentication belongs to this Job.
from pathlib import Path
repo_default = str(Path.cwd().resolve().parents[1])
dbutils.widgets.text("config_dir", repo_default, "Repository root with contract_agent_config.py")
CONFIG_DIR = dbutils.widgets.get("config_dir").strip()
config_path = Path(CONFIG_DIR) / "contract_agent_config.py"
if not config_path.is_file():
    raise FileNotFoundError("Set config_dir to the repository root containing contract_agent_config.py")
# Resolve this exact file, not a stale module cached from another workspace folder.
import importlib.util
spec = importlib.util.spec_from_file_location("aia_ingestion_config", config_path)
cfg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cfg)
w = WorkspaceClient()
HOST = w.config.host.rstrip("/")
WORKSPACE_ID = str(w.get_workspace_id())
DOC_LINK_BASE_URL = f"{HOST}/explore/data/volumes"
CATALOG, TABLES, SOURCE_PATH = cfg.CATALOG, cfg.TABLES, cfg.SOURCE_PATH
VS_ENDPOINT, VS_INDEX = cfg.VS_ENDPOINT, cfg.VS_INDEX
EMBEDDING_DIM, INDEX_COLUMNS = cfg.EMBEDDING_DIM, cfg.INDEX_COLUMNS

EMBEDDING_MODEL = cfg.PIPELINE_EMBEDDING_ENDPOINT   # ai_query endpoint name (the agent uses the gateway name)

PARSED_TABLE        = TABLES["parsed_document"]
METADATA_TABLE      = TABLES["document_metadata"]
CHUNK_TABLE         = TABLES["document_chunk"]
CONTEXT_CHUNK_TABLE = TABLES["contextualized_chunk"]
EMBEDDING_TABLE     = TABLES["chunk_embedding"]

# The supplied pipeline inserts into existing schemas; it does not create these tables.
missing_tables = [name for name in TABLES.values() if not spark.catalog.tableExists(name)]
if missing_tables:
    raise RuntimeError("Create the developer's pipeline table schemas before running ingestion: " + ", ".join(missing_tables))

# ---- Chunking ----
MAX_CHUNK_CHARS = 2000   # ~500 tokens per chunk; tables and very long single elements are never split
EXCLUDED_ELEMENT_TYPES = ("page_header", "page_footer", "page_number")
REBUILD_CHUNKS = False   # True = re-chunk the file even if it was not re-parsed
SUPPORTED_EXT_REGEX = "[.]pdf$"   # only PDF documents are processed

# ---- Document links (one link per PDF file, no page number; base URL + workspace id from config) ----
DOC_URI_EXPR = "regexp_replace(file_path, '^dbfs:', '')"
SOURCE_URI_EXPR = "array_join(transform(split(doc_uri, '/'), s -> replace(url_encode(s), '+', '%20')), '/')"
SOURCE_URL_EXPR = f"""concat('{DOC_LINK_BASE_URL}/',
  element_at(split(doc_uri, '/'), 3), '/', element_at(split(doc_uri, '/'), 4), '/', element_at(split(doc_uri, '/'), 5),
  '?o={WORKSPACE_ID}&volumePath=', replace(url_encode(doc_uri), '+', '%20'))"""
CITATION_EXPR = f"""concat('[', translate(file_name, '[]', '()'), ' | ', translate(coalesce(section_title, 'Document start'), '[]', '()'),
  '](', {SOURCE_URL_EXPR}, ')')"""
# Link columns: written only when the link was opened successfully in Step 3b
LINK_COLS_SQL = f"""CASE WHEN link_ok THEN {SOURCE_URI_EXPR} END AS source_uri,
           CASE WHEN link_ok THEN {SOURCE_URL_EXPR} END AS source_url,
           CASE WHEN link_ok THEN {CITATION_EXPR} END AS citation"""

# ---- ai_extract schema ----
EXTRACT_SCHEMA = """{
  "document_title": {"type": "string", "description": "Title of the document as shown on the first page"},
  "document_type": {"type": "enum", "labels": ["Master Agreement", "License Agreement", "Subscription Agreement", "Order Form", "Pricing Schedule", "Service Schedule", "Amendment", "Addendum", "Purchase Agreement", "Other"], "description": "Type of contract document"},
  "contract_reference": {"type": "string", "description": "Contract, order form or schedule reference number of this document"},
  "parent_agreement_reference": {"type": "string", "description": "Name or reference number of the master or parent agreement this document falls under, if any"},
  "vendor_name": {"type": "string", "description": "Legal name of the vendor, data provider or licensor"},
  "aia_entity": {"type": "string", "description": "Legal name of the AIA contracting entity (client or licensee)"},
  "effective_date": {"type": "string", "description": "Effective or start date in YYYY-MM-DD format"},
  "expiry_date": {"type": "string", "description": "End or expiry date of the term in YYYY-MM-DD format"},
  "signed_date": {"type": "string", "description": "Date the document was signed in YYYY-MM-DD format (latest signature)"},
  "contract_term": {"type": "string", "description": "Duration of the term, e.g. 36 months"},
  "auto_renewal": {"type": "boolean", "description": "True if the contract renews automatically"},
  "renewal_terms": {"type": "string", "description": "Summary of renewal terms and any price uplift on renewal"},
  "termination_notice_period": {"type": "string", "description": "Notice period required to terminate or not renew"},
  "governing_law": {"type": "string", "description": "Governing law jurisdiction"},
  "currency": {"type": "string", "description": "ISO 4217 currency code of the fees"},
  "total_contract_value": {"type": "number", "description": "Total fees payable for the term (or annual fee if only that is stated)"},
  "payment_terms": {"type": "string", "description": "Invoicing frequency and payment terms"},
  "products_services": {"type": "array", "items": {"type": "string"}, "description": "Products, datasets, indices or services licensed"},
  "permitted_users_or_sites": {"type": "string", "description": "Permitted users, locations or usage restrictions"},
  "signatories": {"type": "array", "description": "People who signed the document", "items": {"type": "object", "properties": {"name": {"type": "string"}, "title": {"type": "string"}, "party": {"type": "string"}}}}
}"""
EXTRACT_INSTRUCTIONS = ("These are market data and index vendor contracts (license agreements, order forms, "
                        "pricing schedules, amendments) between AIA entities and vendors such as MSCI, S&P, Fitch, "
                        "Moodys and Cliffwater. Return null when a field is not stated in the document.")
assert "'" not in EXTRACT_SCHEMA + EXTRACT_INSTRUCTIONS  # literals are embedded in SQL

IN_FILES = "file_name IN (SELECT file_name FROM _files)"   # filter used by every step


# ===================================== helpers =====================================
def _sql_str(s):
    """Escape a Python string as a Spark SQL string literal."""
    return "'" + s.replace("\\", "\\\\").replace("'", "\\'") + "'"


def _glob_escape(name):
    """Escape glob characters so READ_FILES reads exactly this file."""
    return re.sub(r"([\[\]{}*?\\])", r"\\\1", name)


def _v(field, cast="STRING"):
    return f"try_cast(ext:response:{field}:value AS {cast})"


def check_document_type(file_name):
    """Check that a document is a PDF: the name must end in .pdf and, if the file exists in SOURCE_PATH,
    its content must start with the PDF signature '%PDF'. Returns (is_pdf, message)."""
    n = file_name.strip().rstrip("/").split("/")[-1]   # accepts a file name or a full /Volumes path
    ext = n.rsplit(".", 1)[-1].lower() if "." in n else ""
    if ext != "pdf":
        return False, (f"Unsupported file format: '{n}' ({'.' + ext if ext else 'no extension'}). "
                       f"Only PDF files are supported.")
    r = requests.get(f"{w.config.host}/api/2.0/fs/files{urllib.parse.quote(SOURCE_PATH + n)}",
                     headers={**w.config.authenticate(), "Range": "bytes=0-4"}, timeout=60)
    if r.status_code in (200, 206) and not r.content.startswith(b"%PDF"):
        return False, f"Unsupported file format: '{n}' has a .pdf extension but is not a valid PDF file."
    return True, f"'{n}' is a PDF"


def set_files(pdf_names):
    """Check each file exists in SOURCE_PATH and register temp view `_files(file_name)`. Returns the names."""
    if isinstance(pdf_names, str):
        pdf_names = [pdf_names]
    names = []
    for n in pdf_names:
        n = n.strip().rstrip("/").split("/")[-1]   # accepts a file name or a full /Volumes path
        if not re.search(SUPPORTED_EXT_REGEX, n.lower()):
            raise ValueError(f"Unsupported file type: {n}")
        try:
            w.files.get_metadata(SOURCE_PATH + n)
        except NotFound:
            raise FileNotFoundError(f"'{n}' not found in {SOURCE_PATH}") from None
        names.append(n)
    if not names:
        raise ValueError("No file name given")
    spark.createDataFrame([(n,) for n in names], "file_name STRING").createOrReplaceTempView("_files")
    return names


def _set_batch(sql):
    """Run a query returning file_path and register the result as temp view `_batch`. Returns the row count."""
    paths = [r.file_path for r in spark.sql(sql).collect()]
    spark.createDataFrame([(p,) for p in paths], "file_path STRING").createOrReplaceTempView("_batch")
    return len(paths)


# ====================================== steps ======================================
def parse_documents(names, force=False):
    """Step 1: ai_parse_document -> PARSED_TABLE. Skips an unchanged file unless force=True."""
    reads = " UNION ALL ".join(
        f"SELECT path, length, modificationTime, content FROM READ_FILES("
        f"{_sql_str(SOURCE_PATH + _glob_escape(n))}, format => 'binaryFile')" for n in names)
    spark.sql(f"""
      CREATE OR REPLACE TEMP VIEW _src_files AS
      SELECT path AS file_path, url_decode(regexp_extract(path, '[^/]+$', 0)) AS file_name,
             length AS file_size, modificationTime AS modification_time, content
      FROM ({reads})""")
    n = _set_batch(f"""
      SELECT s.file_path FROM _src_files s
      LEFT JOIN {PARSED_TABLE} t ON s.file_path = t.file_path AND s.modification_time = t.modification_time
      WHERE {force} OR t.file_path IS NULL""")
    print(f"Files to parse: {n}")
    if n:
        spark.sql(f"DELETE FROM {PARSED_TABLE} WHERE file_path IN (SELECT file_path FROM _batch)")
        spark.sql(f"""
      INSERT INTO {PARSED_TABLE}
      SELECT file_path, file_name, file_size, modification_time, content, parsed_content,
             CASE WHEN parsed_content IS NULL THEN 'ai_parse_document returned NULL'
                  WHEN is_variant_null(parsed_content:error_status) THEN NULL
                  ELSE to_json(parsed_content:error_status) END AS parse_error,
             size(try_cast(parsed_content:document:pages AS ARRAY<VARIANT>))    AS page_count,
             size(try_cast(parsed_content:document:elements AS ARRAY<VARIANT>)) AS element_count,
             current_timestamp() AS parsed_at
      FROM (SELECT s.*, ai_parse_document(s.content, MAP('version', '2.0')) AS parsed_content
            FROM _src_files s JOIN _batch b ON s.file_path = b.file_path)""")


def extract_metadata():
    """Step 2: ai_extract document metadata (only if the file is new or re-parsed) -> METADATA_TABLE."""
    n = _set_batch(f"""
      SELECT p.file_path FROM {PARSED_TABLE} p LEFT JOIN {METADATA_TABLE} m ON p.file_path = m.file_path
      WHERE p.{IN_FILES} AND p.parse_error IS NULL AND (m.file_path IS NULL OR p.parsed_at > m.extracted_at)""")
    print(f"Files to extract: {n}")
    if n:
        spark.sql(f"DELETE FROM {METADATA_TABLE} WHERE file_path IN (SELECT file_path FROM _batch)")
        spark.sql(f"""
      INSERT INTO {METADATA_TABLE}
      SELECT file_path, file_name,
        {_v('document_title')}, {_v('document_type')}, {_v('contract_reference')}, {_v('parent_agreement_reference')},
        {_v('vendor_name')}, {_v('aia_entity')},
        try_to_date({_v('effective_date')}), try_to_date({_v('expiry_date')}), try_to_date({_v('signed_date')}),
        {_v('contract_term')}, {_v('auto_renewal', 'BOOLEAN')}, {_v('renewal_terms')}, {_v('termination_notice_period')},
        {_v('governing_law')}, {_v('currency')}, {_v('total_contract_value', 'DOUBLE')}, {_v('payment_terms')},
        transform(try_cast(ext:response:products_services AS ARRAY<VARIANT>), x -> try_cast(x:value AS STRING)),
        {_v('permitted_users_or_sites')},
        transform(try_cast(ext:response:signatories AS ARRAY<VARIANT>),
                  x -> named_struct('name',  try_cast(x:name:value AS STRING),
                                    'title', try_cast(x:title:value AS STRING),
                                    'party', try_cast(x:party:value AS STRING))),
        ext AS extraction,
        CASE WHEN ext IS NULL THEN 'ai_extract returned NULL'
             WHEN is_variant_null(ext:error_message) THEN NULL
             ELSE try_cast(ext:error_message AS STRING) END AS extraction_error,
        current_timestamp() AS extracted_at
      FROM (SELECT p.file_path, p.file_name,
                   ai_extract(p.parsed_content, '{EXTRACT_SCHEMA}',
                              MAP('version', '2.1', 'enableConfidenceScores', 'true',
                                  'instructions', '{EXTRACT_INSTRUCTIONS}')) AS ext
            FROM {PARSED_TABLE} p JOIN _batch b ON p.file_path = b.file_path)""")


def chunk_documents():
    """Step 3: section-aware chunking (only if the file is new or re-parsed) -> CHUNK_TABLE.
    Headings start a section, tables are their own chunk, text is packed up to MAX_CHUNK_CHARS,
    page headers/footers/numbers and heading-only chunks are dropped."""
    n = _set_batch(f"""
      SELECT p.file_path FROM {PARSED_TABLE} p
      LEFT JOIN (SELECT file_path, max(chunked_at) AS chunked_at FROM {CHUNK_TABLE} WHERE {IN_FILES} GROUP BY file_path) c
        ON p.file_path = c.file_path
      WHERE p.{IN_FILES} AND p.parse_error IS NULL
        AND ({REBUILD_CHUNKS} OR c.file_path IS NULL OR p.parsed_at > c.chunked_at)""")
    print(f"Files to chunk: {n}")
    if not n:
        return
    excluded = ", ".join(f"'{t}'" for t in EXCLUDED_ELEMENT_TYPES)
    spark.sql(f"DELETE FROM {CHUNK_TABLE} WHERE file_path IN (SELECT file_path FROM _batch)")
    spark.sql(f"""
    INSERT INTO {CHUNK_TABLE}
    WITH elems AS (
      SELECT p.file_path, p.file_name, e.pos AS element_idx,
             try_cast(e.el:type AS STRING) AS element_type,
             trim(coalesce(try_cast(e.el:content AS STRING), try_cast(e.el:description AS STRING))) AS content,
             coalesce(try_cast(e.el:bbox[0]:page_id AS INT), 0) AS page_id
      FROM {PARSED_TABLE} p
      JOIN _batch b ON p.file_path = b.file_path
      LATERAL VIEW posexplode(try_cast(p.parsed_content:document:elements AS ARRAY<VARIANT>)) e AS pos, el
    ),
    flagged AS (
      SELECT *,
             CASE WHEN element_type IN ('title', 'section_header') THEN 1 ELSE 0 END AS is_heading,
             CASE WHEN element_type = 'table' THEN 1 ELSE 0 END AS is_table
      FROM elems
      WHERE element_type NOT IN ({excluded}) AND content IS NOT NULL AND content <> ''
    ),
    sectioned AS (
      SELECT *, SUM(is_heading) OVER (PARTITION BY file_path ORDER BY element_idx
                                      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS section_id
      FROM flagged
    ),
    segmented AS (
      SELECT *,
             -- 0 = text before first table, 1 = table 1, 2 = text after table 1, 3 = table 2, ...
             SUM(is_table) OVER (PARTITION BY file_path, section_id ORDER BY element_idx
                                 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) * 2 - is_table AS segment_id,
             first_value(CASE WHEN is_heading = 1 THEN content END, true)
               OVER (PARTITION BY file_path, section_id ORDER BY element_idx
                     ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS section_title
      FROM sectioned
    ),
    seg_arrays AS (
      SELECT file_path,
             aggregate(
               array_sort(collect_list(struct(element_idx, length(content) + 2 AS len))),
               named_struct('cur', 0, 'b', 0,
                            'out', CAST(array() AS ARRAY<STRUCT<element_idx: INT, sub_id: INT>>)),
               (acc, x) -> CASE
                 WHEN acc.cur > 0 AND acc.cur + x.len > {MAX_CHUNK_CHARS}
                   THEN named_struct('cur', x.len, 'b', acc.b + 1,
                          'out', array_append(acc.out, named_struct('element_idx', x.element_idx, 'sub_id', acc.b + 1)))
                 ELSE named_struct('cur', acc.cur + x.len, 'b', acc.b,
                          'out', array_append(acc.out, named_struct('element_idx', x.element_idx, 'sub_id', acc.b)))
               END,
               acc -> acc.out) AS assignments
      FROM segmented
      GROUP BY file_path, section_id, segment_id
    ),
    assigned AS (SELECT file_path, inline(assignments) FROM seg_arrays),
    packed AS (
      SELECT s.*, a.sub_id
      FROM segmented s JOIN assigned a ON s.file_path = a.file_path AND s.element_idx = a.element_idx
    ),
    chunks AS (
      SELECT file_path, first(file_name) AS file_name, section_id, segment_id, sub_id,
             first(section_title) AS section_title,
             concat_ws(char(10) || char(10),
                       transform(array_sort(collect_list(struct(element_idx, content))), x -> x.content)) AS chunk_text,
             min(page_id) + 1 AS page_start, max(page_id) + 1 AS page_end,
             array_distinct(collect_list(element_type)) AS element_types,
             min(element_idx) AS first_element_idx,
             max(is_heading) = 1 AND min(is_heading) = 1 AS heading_only
      FROM packed
      GROUP BY file_path, section_id, segment_id, sub_id
    )
    SELECT concat(md5(file_path), '_', lpad(CAST(chunk_index AS STRING), 4, '0')) AS chunk_id,
           file_path, file_name, chunk_index, CAST(section_id AS INT), coalesce(section_title, 'Document start'),
           page_start, page_end, element_types, chunk_text, length(chunk_text) AS chunk_chars,
           current_timestamp() AS chunked_at,
           {DOC_URI_EXPR} AS doc_uri
    FROM (
      SELECT *, CAST(row_number() OVER (PARTITION BY file_path ORDER BY first_element_idx) - 1 AS INT) AS chunk_index
      FROM chunks WHERE NOT heading_only
    )""")


def validate_doc_links():
    """Step 3b: build one link per file (no page number), open the link and the PDF behind it, and register
    temp view `_link_check(doc_uri, source_url, link_ok, check_detail)`. Only links with link_ok are stored."""
    docs = spark.sql(f"""
      SELECT doc_uri, {SOURCE_URL_EXPR} AS source_url
      FROM (SELECT DISTINCT doc_uri FROM {CHUNK_TABLE} WHERE doc_uri IS NOT NULL AND {IN_FILES})""").collect()
    headers = w.config.authenticate()
    results = []
    for d in docs:
        ok, detail = False, ""
        try:
            vol_path = urllib.parse.parse_qs(urllib.parse.urlparse(d.source_url).query).get("volumePath", [None])[0]
            if not d.source_url.startswith(DOC_LINK_BASE_URL + "/") or vol_path != d.doc_uri:
                detail = f"link does not point to this file: {vol_path}"
            elif (code := requests.get(d.source_url, headers=headers, timeout=60).status_code) != 200:
                detail = f"link returned HTTP {code}"
            else:
                f = requests.get(f"{w.config.host}/api/2.0/fs/files{urllib.parse.quote(d.doc_uri)}",
                                 headers={**headers, "Range": "bytes=0-4"}, timeout=60)
                if f.status_code not in (200, 206):
                    detail = f"file returned HTTP {f.status_code}"
                elif d.doc_uri.lower().endswith(".pdf") and not f.content.startswith(b"%PDF"):
                    detail = "file is not a valid PDF"
                else:
                    ok, detail = True, "ok"
        except Exception as e:
            detail = f"error: {e}"[:300]
        results.append((d.doc_uri, d.source_url, ok, detail))
    spark.createDataFrame(results, "doc_uri STRING, source_url STRING, link_ok BOOLEAN, check_detail STRING") \
         .createOrReplaceTempView("_link_check")
    n_ok = sum(r[2] for r in results)
    print(f"Links checked: {len(results)} | opened OK: {n_ok} | failed: {len(results) - n_ok}")
    for r in results:
        if not r[2]:
            print(f"  FAILED {r[0]}: {r[3]}")


def contextualize_chunks():
    """Step 4: chunks + metadata -> CONTEXT_CHUNK_TABLE (only if chunks or metadata changed)."""
    n = _set_batch(f"""
      SELECT c.file_path
      FROM (SELECT file_path, max(chunked_at) AS chunked_at FROM {CHUNK_TABLE} WHERE {IN_FILES} GROUP BY file_path) c
      LEFT JOIN {METADATA_TABLE} m ON c.file_path = m.file_path
      LEFT JOIN (SELECT file_path, max(created_at) AS created_at FROM {CONTEXT_CHUNK_TABLE} WHERE {IN_FILES}
                 GROUP BY file_path) x ON c.file_path = x.file_path
      WHERE x.file_path IS NULL OR c.chunked_at > x.created_at OR m.extracted_at > x.created_at""")
    print(f"Files to contextualize: {n}")
    if not n:
        return
    spark.sql(f"DELETE FROM {CONTEXT_CHUNK_TABLE} WHERE file_path IN (SELECT file_path FROM _batch)")
    spark.sql(f"""
    INSERT INTO {CONTEXT_CHUNK_TABLE}
    SELECT chunk_id, file_path, file_name, chunk_index, section_title, page_start, page_end, element_types,
           document_title, document_type, contract_reference, vendor_name, aia_entity,
           effective_date, expiry_date, products_services,
           context_header, chunk_text,
           concat(context_header, char(10), char(10), chunk_text) AS contextualized_text,
           current_timestamp() AS created_at,
           doc_uri,
           {LINK_COLS_SQL}
    FROM (
      SELECT c.*, coalesce(lk.link_ok, false) AS link_ok, m.document_title, m.document_type, m.contract_reference,
             m.vendor_name, m.aia_entity, m.effective_date, m.expiry_date, m.products_services,
             concat_ws(char(10),
               concat('Document: ', coalesce(m.document_title, c.file_name), ' (', c.file_name, ')'),
               concat('Document type: ', m.document_type),
               concat('Vendor: ', m.vendor_name),
               concat('AIA entity: ', m.aia_entity),
               concat('Contract reference: ', m.contract_reference),
               concat('Parent agreement: ', m.parent_agreement_reference),
               concat('Term: ', CAST(m.effective_date AS STRING), ' to ', CAST(m.expiry_date AS STRING)),
               concat('Products/services: ', nullif(array_join(m.products_services, ', '), '')),
               concat('Section: ', c.section_title),
               concat('Pages: ', CAST(c.page_start AS STRING), '-', CAST(c.page_end AS STRING)),
               concat('Source URI: ', c.doc_uri)
             ) AS context_header
      FROM {CHUNK_TABLE} c
      JOIN _batch b ON c.file_path = b.file_path
      LEFT JOIN {METADATA_TABLE} m ON c.file_path = m.file_path
      LEFT JOIN _link_check lk ON c.doc_uri = lk.doc_uri
    )""")


def refresh_links():
    """Step 4b: re-apply the validated links to the file's existing rows (only changed rows are rewritten)."""
    m = spark.sql(f"""
    MERGE INTO {CONTEXT_CHUNK_TABLE} t
    USING (SELECT chunk_id, {LINK_COLS_SQL}
           FROM (SELECT c.*, coalesce(lk.link_ok, false) AS link_ok
                 FROM {CONTEXT_CHUNK_TABLE} c LEFT JOIN _link_check lk ON c.doc_uri = lk.doc_uri
                 WHERE c.{IN_FILES})) s
    ON t.chunk_id = s.chunk_id
    WHEN MATCHED AND (t.source_uri IS DISTINCT FROM s.source_uri OR t.source_url IS DISTINCT FROM s.source_url
                      OR t.citation IS DISTINCT FROM s.citation)
      THEN UPDATE SET t.source_uri = s.source_uri, t.source_url = s.source_url, t.citation = s.citation""").first()
    print(f"Link rows updated: {m['num_updated_rows']}")


def embed_chunks():
    """Step 5: sync the file's rows into EMBEDDING_TABLE and embed only new/changed text
    (metadata-only change keeps the embedding; chunks that no longer exist are deleted)."""
    cols_sql = ", ".join(INDEX_COLUMNS)
    spark.sql(f"""
      CREATE OR REPLACE TEMP VIEW _emb_src AS
      SELECT {cols_sql}, md5(to_json(struct({cols_sql}))) AS row_hash
      FROM {CONTEXT_CHUNK_TABLE} WHERE {IN_FILES}""")
    set_clause = ", ".join(f"t.{c} = s.{c}" for c in INDEX_COLUMNS if c != "chunk_id")
    m = spark.sql(f"""
      MERGE INTO {EMBEDDING_TABLE} t
      USING _emb_src s ON t.chunk_id = s.chunk_id
      WHEN MATCHED AND t.row_hash IS DISTINCT FROM s.row_hash THEN UPDATE SET
        {set_clause}, t.row_hash = s.row_hash,
        t.embedding   = CASE WHEN t.contextualized_text IS DISTINCT FROM s.contextualized_text THEN NULL ELSE t.embedding END,
        t.embedded_at = CASE WHEN t.contextualized_text IS DISTINCT FROM s.contextualized_text THEN NULL ELSE t.embedded_at END
      WHEN NOT MATCHED THEN INSERT ({cols_sql}, row_hash) VALUES ({", ".join(f"s.{c}" for c in INDEX_COLUMNS)}, s.row_hash)
    """).first()
    d = spark.sql(f"""DELETE FROM {EMBEDDING_TABLE}
                      WHERE {IN_FILES} AND chunk_id NOT IN (SELECT chunk_id FROM _emb_src)""").first()
    print(f"Synced rows -> inserted: {m['num_inserted_rows']}, updated: {m['num_updated_rows']}, "
          f"deleted: {d['num_affected_rows']}")

    to_embed = spark.sql(f"SELECT count(*) AS n FROM {EMBEDDING_TABLE} WHERE embedding IS NULL AND {IN_FILES}").first().n
    print(f"Chunks to embed: {to_embed}")
    if to_embed:
        # ai_query can't run inside UPDATE/MERGE: compute once into a staging table, then MERGE it in
        staging = f"{EMBEDDING_TABLE}_staging"
        spark.sql(f"""
          CREATE OR REPLACE TABLE {staging} AS
          SELECT chunk_id, row_hash, ai_query('{EMBEDDING_MODEL}', contextualized_text) AS embedding
          FROM {EMBEDDING_TABLE} WHERE embedding IS NULL AND {IN_FILES}""")
        spark.sql(f"""
          MERGE INTO {EMBEDDING_TABLE} t USING {staging} s
            ON t.chunk_id = s.chunk_id AND t.row_hash = s.row_hash
          WHEN MATCHED THEN UPDATE SET t.embedding = s.embedding, t.embedded_at = current_timestamp()""")
        spark.sql(f"DROP TABLE IF EXISTS {staging}")


def _index_status():
    st = w.vector_search_indexes.get_index(VS_INDEX).status
    done = st.triggered_update_status.last_processed_commit_version if st.triggered_update_status else None
    return st, (done if done is not None else -1)


def sync_vector_index():
    """Step 6: incremental sync of VS_INDEX (endpoint VS_ENDPOINT); skipped when the index is already up to date."""
    expected = spark.table(EMBEDDING_TABLE).count()
    last_version = spark.sql(f"DESCRIBE HISTORY {EMBEDDING_TABLE} LIMIT 1").first().version
    try:
        st, done = _index_status()
    except (NotFound, ResourceDoesNotExist):
        print(f"Creating index {VS_INDEX} on endpoint {VS_ENDPOINT}")
        w.vector_search_indexes.create_index(
            name=VS_INDEX, endpoint_name=VS_ENDPOINT, primary_key="chunk_id",
            index_type=vs.VectorIndexType.DELTA_SYNC,
            delta_sync_index_spec=vs.DeltaSyncVectorIndexSpecRequest(
                source_table=EMBEDDING_TABLE,
                embedding_vector_columns=[vs.EmbeddingVectorColumn(name="embedding", embedding_dimension=EMBEDDING_DIM)],
                columns_to_sync=INDEX_COLUMNS, pipeline_type=vs.PipelineType.TRIGGERED))
        st, done = None, -1
    if st is not None and st.ready and done >= last_version and (st.indexed_row_count or 0) == expected:
        print(f"Index already up to date ({expected} rows, table version {last_version}); sync skipped")
        return
    if st is not None:
        for _ in range(30):   # a sync can only start when no other sync is running
            try:
                w.vector_search_indexes.sync_index(VS_INDEX)
                print(f"Sync triggered: {VS_INDEX}")
                break
            except BadRequest as e:
                if "currently running" not in str(e):
                    raise
                time.sleep(20)
    for _ in range(90):
        st, done = _index_status()
        rows = st.indexed_row_count or 0
        if st.ready and rows == expected and done >= last_version:
            print(f"Index synced: {rows}/{expected} rows, table version {done}")
            return
        print(f"  waiting... ready={st.ready} indexed_rows={rows}/{expected} version={done}/{last_version}")
        time.sleep(20)
    print("Index not fully synced yet - run sync_vector_index() again later to check status.")


def _summary():
    """One row per file showing what is stored in every pipeline table."""
    return spark.sql(f"""
      SELECT f.file_name, p.page_count, p.parse_error, m.document_type, m.vendor_name, m.extraction_error,
             c.chunks, x.context_chunks, x.chunks_with_link, x.source_url,
             e.embedded_chunks, e.missing_embeddings
      FROM _files f
      LEFT JOIN {PARSED_TABLE} p ON p.file_name = f.file_name
      LEFT JOIN {METADATA_TABLE} m ON m.file_name = f.file_name
      LEFT JOIN (SELECT file_name, count(*) AS chunks FROM {CHUNK_TABLE} WHERE {IN_FILES} GROUP BY file_name) c
        ON c.file_name = f.file_name
      LEFT JOIN (SELECT file_name, count(*) AS context_chunks, count(source_url) AS chunks_with_link,
                        first(source_url, true) AS source_url
                 FROM {CONTEXT_CHUNK_TABLE} WHERE {IN_FILES} GROUP BY file_name) x ON x.file_name = f.file_name
      LEFT JOIN (SELECT file_name, count(*) AS embedded_chunks, count_if(embedding IS NULL) AS missing_embeddings
                 FROM {EMBEDDING_TABLE} WHERE {IN_FILES} GROUP BY file_name) e ON e.file_name = f.file_name""")


# ================================== run_pipeline ===================================
def run_pipeline(pdf_name, force=False):
    """Run every step on one PDF (file name in SOURCE_PATH or full /Volumes path; a list also works).
    force=True re-parses the file even if unchanged; the later steps then rebuild automatically.
    Each document is first checked with check_document_type(); non-PDF files are not processed and
    an 'Unsupported file format' message is returned."""
    items = [pdf_name] if isinstance(pdf_name, str) else list(pdf_name)
    pdfs, rejected = [], []
    for item in items:
        is_pdf, msg = check_document_type(item)
        (pdfs if is_pdf else rejected).append(item if is_pdf else msg)
    if not pdfs:
        return "\n".join(rejected) if rejected else "No file name given"
    for msg in rejected:   # list input with some non-PDF files: skip them, process the PDFs
        print(f"Skipped - {msg}")
    names = set_files(pdfs)
    print(f"Files: {names}")
    steps = [("Step 1: Parse documents", lambda: parse_documents(names, force=force)),
             ("Step 2: Extract metadata", extract_metadata),
             ("Step 3: Chunk documents", chunk_documents),
             ("Step 3b: Validate PDF links", validate_doc_links),
             ("Step 4: Contextualize chunks", contextualize_chunks),
             ("Step 4b: Refresh links", refresh_links),
             ("Step 5: Embed chunks", embed_chunks),
             ("Step 6: Sync Vector Search index", sync_vector_index)]
    t0 = time.time()
    for name, fn in steps:
        s = time.time()
        print(f"\n===== {name} =====")
        fn()
        print(f"-- done in {time.time() - s:.0f}s")
    print(f"\nPipeline finished in {time.time() - t0:.0f}s")
    display(_summary())

print("Pipeline ready: call run_pipeline('<pdf file name>')")

# File de-duplication + hand-off to `run_pipeline`
The cells above are the **Parsing and Indexing Pipeline**, copied unchanged from `/Workspace/AIA_contract_dir/Rohit/Parsing and Indexing Pipeline`.
The cells below are the **dedup code** (settings come from `dedup_config.json`) and the glue that passes `files_to_index` to `run_pipeline`.

* `run_pipeline` only processes PDFs located directly in `.../MO_Contract_Mgt/Input/` (not subfolders). The glue stops with a clear error otherwise.
* `run_pipeline` has no remove step, so `files_to_remove` is appended to `files_pending_removal` in the dedup schema.

In [ ]:
import os

# The pipeline notebook defines SOURCE_PATH with a trailing slash ('/Volumes/.../Input/') and its functions read it
# at call time. The dedup cell below redefines SOURCE_PATH (no trailing slash), so keep the pipeline's value here
# and put it back only while run_pipeline runs.
PIPELINE_SOURCE_PATH = SOURCE_PATH

def run_dedup_handoff(files_to_index, files_to_remove):
    """Hook called by the dedup cell. Calls run_pipeline() as-is. If this raises, the dedup cell does not record
    the hand-off, and the same files are listed again on the next run."""
    global SOURCE_PATH
    dedup_source_path = SOURCE_PATH
    pipeline_root = PIPELINE_SOURCE_PATH.rstrip("/")

    if dedup_source_path.rstrip("/") != pipeline_root:
        raise ValueError(f"Dedup source_path '{dedup_source_path}' must match the pipeline SOURCE_PATH '{PIPELINE_SOURCE_PATH}'.")

    # run_pipeline resolves every file as SOURCE_PATH + file name, so files in subfolders cannot be processed
    # (and could be confused with a same-named file in the root). Stop before anything runs.
    in_subfolders = [p for p in files_to_index if os.path.dirname(p) != pipeline_root]
    if in_subfolders:
        raise ValueError("run_pipeline only reads files directly in " + PIPELINE_SOURCE_PATH +
                         ". Move these files there or remove them:\n" + "\n".join(in_subfolders))

    # run_pipeline has no remove step: keep the list so the removal can be done later
    if files_to_remove:
        (spark.createDataFrame([(p,) for p in files_to_remove], "file_path STRING")
              .selectExpr("file_path", "current_timestamp() AS listed_at")
              .write.mode("append").saveAsTable(qualify(f"{CATALOG}.{SCHEMA}.files_pending_removal")))

    if files_to_index:
        SOURCE_PATH = PIPELINE_SOURCE_PATH
        try:
            run_pipeline(files_to_index)
        finally:
            SOURCE_PATH = dedup_source_path

In [ ]:
# File de-duplication (any format) before parse / chunk / index.
#   Case 1: same name, changed content     -> keep the latest version
#   Case 2: same content, different name   -> keep the latest valid file
# Outputs files_to_index / files_to_remove, hands them to the downstream function, then records the hand-off
# in file_registry so the next run lists only new changes. No prints or displays: the result is returned via notebook exit.
import os, re, json, hashlib
from concurrent.futures import ThreadPoolExecutor
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType

# ---------------- Parameters (from the config file) ----------------
# All settings live in dedup_config.json. Only the config file path is a parameter (a Job can override it per environment).
dbutils.widgets.text("config_path", str(Path(CONFIG_DIR) / "setup" / "ingestion" / "dedup_config.json"), "Config file")
CONFIG_PATH = dbutils.widgets.get("config_path").strip()
with open(CONFIG_PATH) as fh:
    CONFIG = json.load(fh)

# Downstream parse/chunk/index function from the other team (glue in the previous cell calls run_pipeline)
DOWNSTREAM_FN = run_dedup_handoff

_IDENT = re.compile(r"^[A-Za-z0-9_\-]+$")
def qualify(name: str) -> str:
    """Validate a 1-3 part identifier and return it backtick-quoted (prevents SQL injection via parameters)."""
    parts = name.strip().split(".")
    if not 1 <= len(parts) <= 3 or not all(_IDENT.match(p) for p in parts):
        raise ValueError(f"Invalid identifier: {name!r}")
    return ".".join(f"`{p}`" for p in parts)

SOURCE_PATH      = str(CONFIG["source_path"]).strip().rstrip("/")
CATALOG          = str(CONFIG["catalog"]).strip()
SCHEMA           = str(CONFIG["schema"]).strip()
SIMULATE_HANDOFF = str(CONFIG.get("simulate_handoff", False)).strip().lower() == "true"
HASH_WORKERS     = int(CONFIG.get("hash_workers", 16))
INCLUDE_EXT      = {"." + str(e).strip().lower().lstrip(".") for e in CONFIG.get("file_extensions", []) if str(e).strip()}

if not CATALOG or not SCHEMA:
    raise ValueError("Parameters 'catalog' and 'schema' are required.")
if not (SOURCE_PATH.startswith("/Volumes/") or SOURCE_PATH.startswith("/Workspace/")):
    raise ValueError("source_path must start with /Volumes/ or /Workspace/")

REGISTRY   = qualify(f"{CATALOG}.{SCHEMA}.file_registry")
TO_PROCESS = qualify(f"{CATALOG}.{SCHEMA}.files_to_process")
TO_REMOVE  = qualify(f"{CATALOG}.{SCHEMA}.files_to_remove")

# ---------------- 1. Registry table (memory between runs) ----------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {REGISTRY} (
  file_path         STRING    COMMENT 'Full path of the file (primary key)',
  file_name         STRING,
  file_size         BIGINT,
  mtime_epoch_ms    BIGINT    COMMENT 'File modification time in epoch ms (used to skip re-hashing)',
  modification_time TIMESTAMP,
  content_hash      STRING    COMMENT 'SHA-256 of file bytes',
  is_valid          BOOLEAN   COMMENT 'Non-empty and bytes match the extension',
  status            STRING    COMMENT 'ACTIVE | DUPLICATE_CONTENT | SUPERSEDED_NAME | INVALID | DELETED',
  duplicate_of      STRING    COMMENT 'File path of the kept file when this one is a duplicate',
  processed_hash    STRING    COMMENT 'content_hash last handed off downstream (NULL = not handed off / removed)',
  first_seen_at     TIMESTAMP,
  last_seen_at      TIMESTAMP,
  updated_at        TIMESTAMP
)
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

# ---------------- 2. Scan folder (metadata only) ----------------
SKIP_PREFIXES = (".", "~$")                      # hidden files, Office lock files
SKIP_SUFFIXES = (".tmp", ".part", ".crdownload")  # temp / incomplete uploads
ZIP_TYPES = {".docx", ".xlsx", ".pptx", ".zip", ".odt", ".ods", ".odp", ".epub"}
OLE_TYPES = {".doc", ".xls", ".ppt", ".msg"}

def is_candidate(name: str) -> bool:
    low = name.lower()
    if low.startswith(SKIP_PREFIXES) or low.endswith(SKIP_SUFFIXES):
        return False
    return not INCLUDE_EXT or os.path.splitext(low)[1] in INCLUDE_EXT

def list_files(root: str):
    rows = []
    for dirpath, _, names in os.walk(root):
        for n in names:
            if is_candidate(n):
                p = os.path.join(dirpath, n)
                try:
                    st = os.stat(p)
                except OSError:
                    continue
                rows.append({"file_path": p, "file_name": n,
                             "file_size": int(st.st_size), "mtime_epoch_ms": int(st.st_mtime * 1000)})
    return rows

def is_valid_file(name: str, size: int, head: bytes, tail: bytes) -> bool:
    if size == 0:
        return False
    ext = os.path.splitext(name.lower())[1]
    if ext == ".pdf":
        return head.startswith(b"%PDF-") and b"%%EOF" in tail
    if ext in ZIP_TYPES:
        return head.startswith(b"PK\x03\x04")
    if ext in OLE_TYPES:
        return head.startswith(b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1")
    if ext == ".png":
        return head.startswith(b"\x89PNG")
    if ext in (".jpg", ".jpeg"):
        return head.startswith(b"\xff\xd8\xff")
    return True

def hash_and_validate(path: str, block: int = 8 * 1024 * 1024):
    """Streamed SHA-256 + validity check. Unreadable files return (None, False) and become INVALID."""
    h, head, tail, size = hashlib.sha256(), b"", b"", 0
    try:
        with open(path, "rb") as fh:
            while True:
                chunk = fh.read(block)
                if not chunk:
                    break
                if len(head) < 8:
                    head = (head + chunk)[:8]
                h.update(chunk)
                size += len(chunk)
                tail = (tail + chunk[-2048:])[-2048:]
        return h.hexdigest(), is_valid_file(os.path.basename(path), size, head, tail)
    except Exception:
        return None, False

files = list_files(SOURCE_PATH)
if not files:
    # Safety guard: an empty listing (wrong path / permissions) would otherwise mark every file as DELETED
    raise RuntimeError(f"No files found under {SOURCE_PATH}. Stopping so the registry is not wiped.")

# ---------------- 3. Hash only new or changed files ----------------
known = {r["file_path"]: r for r in spark.table(REGISTRY)
         .select("file_path", "file_size", "mtime_epoch_ms", "content_hash", "is_valid").collect()}
to_hash = []
for f in files:
    k = known.get(f["file_path"])
    if k and k["content_hash"] and k["file_size"] == f["file_size"] and k["mtime_epoch_ms"] == f["mtime_epoch_ms"]:
        f["content_hash"], f["is_valid"] = k["content_hash"], bool(k["is_valid"])
    else:
        to_hash.append(f)
with ThreadPoolExecutor(max_workers=HASH_WORKERS) as ex:
    for f, (hsh, ok) in zip(to_hash, ex.map(lambda x: hash_and_validate(x["file_path"]), to_hash)):
        f["content_hash"], f["is_valid"] = hsh, ok

_schema = StructType([
    StructField("file_path", StringType()), StructField("file_name", StringType()),
    StructField("file_size", LongType()), StructField("mtime_epoch_ms", LongType()),
    StructField("content_hash", StringType()), StructField("is_valid", BooleanType()),
])
spark.createDataFrame(
    [(f["file_path"], f["file_name"], f["file_size"], f["mtime_epoch_ms"], f["content_hash"], f["is_valid"]) for f in files],
    _schema).createOrReplaceTempView("current_files")

# ---------------- 4. Dedup rules ----------------
#   INVALID           : empty / unreadable / bytes don't match extension (never kept)
#   DUPLICATE_CONTENT : same content hash as a newer valid file (case 2)
#   SUPERSEDED_NAME   : same file name as a newer file in another folder (case 1)
#   Same path + changed bytes (case 1): hash changes -> listed as CONTENT_CHANGED
spark.sql("""
WITH base AS (
  SELECT *, timestamp_millis(mtime_epoch_ms) AS modification_time, lower(file_name) AS name_key
  FROM current_files
),
content_rank AS (
  SELECT file_path, name_key, mtime_epoch_ms,
         row_number() OVER w_c AS rn_content,
         first_value(file_path) OVER w_c AS content_winner
  FROM base
  WHERE is_valid AND content_hash IS NOT NULL
  WINDOW w_c AS (PARTITION BY content_hash ORDER BY mtime_epoch_ms DESC, file_path)
),
name_rank AS (
  SELECT file_path,
         row_number() OVER w_n AS rn_name,
         first_value(file_path) OVER w_n AS name_winner
  FROM content_rank
  WHERE rn_content = 1
  WINDOW w_n AS (PARTITION BY name_key ORDER BY mtime_epoch_ms DESC, file_path)
)
SELECT b.file_path, b.file_name, b.file_size, b.mtime_epoch_ms, b.modification_time,
       b.content_hash, b.is_valid,
       CASE WHEN NOT b.is_valid OR b.content_hash IS NULL THEN 'INVALID'
            WHEN c.rn_content > 1 THEN 'DUPLICATE_CONTENT'
            WHEN n.rn_name    > 1 THEN 'SUPERSEDED_NAME'
            ELSE 'ACTIVE' END AS status,
       CASE WHEN c.rn_content > 1 THEN c.content_winner
            WHEN n.rn_name    > 1 THEN n.name_winner END AS duplicate_of
FROM base b
LEFT JOIN content_rank c ON b.file_path = c.file_path
LEFT JOIN name_rank    n ON b.file_path = n.file_path
""").createOrReplaceTempView("resolved_files")

# ---------------- 5. Upsert registry (processed_hash untouched here) ----------------
spark.sql(f"""
MERGE INTO {REGISTRY} t
USING resolved_files s
ON t.file_path = s.file_path
WHEN MATCHED THEN UPDATE SET
  file_name = s.file_name, file_size = s.file_size, mtime_epoch_ms = s.mtime_epoch_ms,
  modification_time = s.modification_time, content_hash = s.content_hash, is_valid = s.is_valid,
  status = s.status, duplicate_of = s.duplicate_of,
  last_seen_at = current_timestamp(), updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT
  (file_path, file_name, file_size, mtime_epoch_ms, modification_time, content_hash, is_valid,
   status, duplicate_of, processed_hash, first_seen_at, last_seen_at, updated_at)
  VALUES (s.file_path, s.file_name, s.file_size, s.mtime_epoch_ms, s.modification_time, s.content_hash, s.is_valid,
   s.status, s.duplicate_of, NULL, current_timestamp(), current_timestamp(), current_timestamp())
WHEN NOT MATCHED BY SOURCE AND t.status <> 'DELETED' AND startswith(t.file_path, :root) THEN UPDATE SET
  status = 'DELETED', duplicate_of = NULL, updated_at = current_timestamp()
""", args={"root": SOURCE_PATH + "/"})

# ---------------- 6. Build the two lists ----------------
spark.sql(f"""
CREATE OR REPLACE TABLE {TO_PROCESS} AS
SELECT file_path, file_name, content_hash, modification_time,
       CASE WHEN processed_hash IS NULL THEN 'NEW' ELSE 'CONTENT_CHANGED' END AS reason
FROM {REGISTRY}
WHERE status = 'ACTIVE' AND (processed_hash IS NULL OR processed_hash <> content_hash)
""")
spark.sql(f"""
CREATE OR REPLACE TABLE {TO_REMOVE} AS
SELECT file_path, file_name, status AS reason, duplicate_of AS replaced_by
FROM {REGISTRY}
WHERE status <> 'ACTIVE' AND processed_hash IS NOT NULL
""")
files_to_index  = [r.file_path for r in spark.table(TO_PROCESS).select("file_path").collect()]
files_to_remove = [r.file_path for r in spark.table(TO_REMOVE).select("file_path").collect()]

# ---------------- 7. Hand off and record ----------------
def record_handoff():
    # Indexed files: processed_hash = current hash (only if content did not change again meanwhile)
    spark.sql(f"""
    MERGE INTO {REGISTRY} t USING {TO_PROCESS} s
    ON t.file_path = s.file_path AND t.content_hash = s.content_hash
    WHEN MATCHED THEN UPDATE SET processed_hash = s.content_hash, updated_at = current_timestamp()
    """)
    # Removed files: processed_hash = NULL, so they are not listed for removal again
    spark.sql(f"""
    MERGE INTO {REGISTRY} t USING {TO_REMOVE} s
    ON t.file_path = s.file_path AND t.status <> 'ACTIVE'
    WHEN MATCHED THEN UPDATE SET processed_hash = NULL, updated_at = current_timestamp()
    """)

handed_off = False
if files_to_index or files_to_remove:
    if DOWNSTREAM_FN is not None:
        # If this raises, the run fails, nothing is recorded, and the same files are listed again next run
        DOWNSTREAM_FN(files_to_index=files_to_index, files_to_remove=files_to_remove)
        record_handoff()
        handed_off = True
    elif SIMULATE_HANDOFF:
        record_handoff()
        handed_off = True

result = {"files_scanned": len(files), "files_hashed": len(to_hash),
          "files_to_index": len(files_to_index), "files_to_remove": len(files_to_remove),
          "handed_off": handed_off}
dbutils.jobs.taskValues.set(key="dedup_result", value=result)  # readable by downstream Job tasks
dbutils.notebook.exit(json.dumps(result))